# Ảnh lóe Wu et al. (ICCV 2021) → Kaggle dataset

Notebook này **không train**. Nó tải bộ ảnh **chỉ có lóe** của Wu, He, Xue, Garg, Chen, Veeraraghavan, Barron,
*How to Train Neural Networks for Flare Removal* (ICCV 2021) từ bucket công khai `gs://gresearch/lens-flare`, kiểm
MD5 từng file với bucket, thu cạnh dài về 640 px, rồi tạo Kaggle dataset `<tài khoản>/wu-lens-flare-640`.
Sau đó notebook train (p45_sunflare trở đi) chỉ cần *Add Input*: Cell 4 của `qwt-jaco-jepa-ijepa.ipynb` tự tìm thấy
nó qua `find_wu_flares` và không tải lại nữa.

| | |
|---|---|
| nguồn | `https://storage.googleapis.com/gresearch/lens-flare/{captured,simulated}/*.png`, tải qua HTTPS, không cần đăng nhập |
| nội dung (đo 20/10/2026) | `captured/` 2 001 ảnh chụp thật (1,27 GB), `simulated/` 3 000 ảnh mô phỏng (1,23 GB), đều 1008×752 RGB 8 bit trên nền tối |
| kiểm | mỗi file tải về được so cỡ và MD5 với bucket (lệch thì tải lại, tối đa 4 lần), rồi giải mã lại ảnh đã ghi |
| thu nhỏ | `SIDE = 640`: cạnh dài về 640 px (640×477), bộ lọc BOX trên ánh sáng tuyến tính (gamma 2,2), giữ tổng ánh sáng. Giống từng pixel với `qjepa/data/wu_flares.py:shrink`, tức bản mà notebook train tự tải. `SIDE = None` giữ nguyên cỡ gốc |
| vuông | `SQUARE = True`: đệm trên dưới thành **640×640** bằng giá trị nhỏ nhất của từng kênh (nền tối của chính ảnh lóe), ảnh ở giữa, không kéo giãn, không mất pixel. Đúng phép đệm mà `corruptions/lens_flare.py` làm khi nạp ảnh không vuông, nên với repo ảnh 640×640 và 640×477 cho cùng một lớp lóe |
| cấu trúc | `captured/`, `simulated/` (PNG 640×640), `SOURCE.txt`, `LICENSE` (CC BY 4.0 nguyên văn), `README.md`. Tên file giữ nguyên |
| giấy phép | **CC BY 4.0** (README của `google-research/flare_removal`): được phân phối lại, kể cả public, miễn ghi nguồn. License trên Kaggle là `CC-BY-4.0` |

## Chạy
1. *Settings*: **Accelerator: None** (CPU), **Internet: On**.
2. *Add-ons → Secrets*: **`KAGGLE_API_TOKEN`** (kaggle.com → Settings → API → *Generate New Token*), bật *Attached*
   cho chính notebook này. Cặp cũ `KAGGLE_USERNAME` + `KAGGLE_KEY` cũng được. Khoá chỉ nằm trong biến môi trường của
   tiến trình con `kaggle`, không in ra, không ghi file.
   **Không có secret thì notebook vẫn chạy hết**: Cell 7 đóng gói dataset vào `/kaggle/working`. Sau đó gắn secret,
   chạy Cell 1, 2, 3 rồi **Cell 9** để tạo dataset từ gói đó (không tải lại ảnh), hoặc tạo tay bằng tab
   *Output → New Dataset*.
3. *Save Version → Save & Run All (Commit)*. Dữ liệu trung gian nằm trong `/tmp`.

Thời gian chưa đo trên Kaggle; ước tính 10–25 phút (tải 2,5 GB, thu nhỏ 5 001 ảnh, upload ~1 GB).

## Cell 1 — Lựa chọn

`PUBLIC = False` tạo dataset riêng tư (notebook train của chính tài khoản vẫn gắn được). Giấy phép cho phép public:
đổi thành `True` nếu muốn, và chỉ có tác dụng khi Cell 4–5 không còn vấn đề.

In [ ]:
from pathlib import Path

BUCKET = 'https://storage.googleapis.com/gresearch'
LISTING_URL = 'https://storage.googleapis.com/storage/v1/b/gresearch/o'
README_URL = 'https://raw.githubusercontent.com/google-research/google-research/master/flare_removal/README.md'
LICENSE_URL = 'https://creativecommons.org/licenses/by/4.0/legalcode.txt'
PARTS = ('captured', 'simulated')
EXPECTED = {'captured': 2001, 'simulated': 3000}

SIDE = 640           # cạnh dài sau khi thu nhỏ, như fetch_wu_flares(side=640) của repo; None = giữ cỡ gốc
SQUARE = True        # đệm thành ảnh vuông SIDE×SIDE bằng nền của ảnh (như lens_flare.py khi nạp); False = giữ 640×477
if SIDE:
    SLUG = f'wu-lens-flare-{SIDE}'
    TITLE = f'Wu et al. lens flares, {SIDE}x{SIDE}' if SQUARE else f'Wu et al. lens flares, {SIDE} px'
    SUBTITLE = f'5001 flare-only images (ICCV 2021), {f"{SIDE}x{SIDE}" if SQUARE else f"long side {SIDE} px"}, CC BY 4.0'
else:
    SLUG = 'wu-lens-flare'
    TITLE = 'Wu et al. lens flares (square)' if SQUARE else 'Wu et al. lens flares (mirror)'
    SUBTITLE = '5001 flare-only images from How to Train Neural Networks for Flare Removal'
PUBLIC = False       # chỉ có tác dụng khi Cell 4 và Cell 5 không còn vấn đề
KAGGLE_OWNER = None  # None = tài khoản của khoá

OUT = Path('/tmp/wu_flare')                # ảnh tải về (đã thu nhỏ), {captured,simulated}/
UPLOAD = Path('/tmp/upload/wu_flare')
OUTPUT_DIR = Path('/kaggle/working')       # chỉ dùng khi không có khoá: Cell 7 đóng gói dataset vào đây
REPORT = OUTPUT_DIR / 'wu_flare_report.md'

WORKERS = 16          # luồng tải song song
RETRIES = 4           # mỗi file: số lần thử khi lỗi mạng hoặc lệch MD5
GAMMA = 2.2           # như qjepa/data/wu_flares.py

## Cell 2 — Hàm dùng chung (chỉ định nghĩa, chưa chạy gì)

In [ ]:
import base64
import hashlib
import json
import os
import posixpath
import shutil
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO

import numpy as np
from PIL import Image


def gb(n):
    return f'{n / 1e9:.2f} GB'


def run(command, env=None, timeout=None):
    """(mã thoát, stdout, stderr) của một tiến trình con."""
    result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=timeout)
    return result.returncode, result.stdout, result.stderr


def kaggle(*args, env, timeout=None):
    code, out, err = run([shutil.which('kaggle') or 'kaggle', *args], env=env, timeout=timeout)
    return code, (out + err).strip()


def list_part(part):
    """[(tên, byte, md5 base64)] các PNG của ``part`` trong bucket (API JSON của Cloud Storage, phân trang)."""
    rows, token = [], None
    while True:
        url = (f'{LISTING_URL}?prefix=lens-flare/{part}/&fields=items(name,size,md5Hash),nextPageToken'
               f'&maxResults=1000' + (f'&pageToken={token}' if token else ''))
        with urllib.request.urlopen(url, timeout=60) as response:
            data = json.load(response)
        rows += [(item['name'].rsplit('/', 1)[-1], int(item['size']), item['md5Hash'])
                 for item in data.get('items', []) if item['name'].endswith('.png')]
        token = data.get('nextPageToken')
        if not token:
            return sorted(rows)


def shrink(pixels, side):
    """Cạnh dài về ``side`` px, BOX trên ánh sáng tuyến tính (gamma 2,2), trả về uint8; nhỏ hơn thì giữ nguyên.
    Chép nguyên từ qjepa/data/wu_flares.py để dataset trùng từng pixel với bản notebook train tự tải."""
    height, width = pixels.shape[:2]
    if max(height, width) <= side:
        return pixels
    scale = side / max(height, width)
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    linear = np.power(pixels.astype(np.float32) / 255.0, GAMMA)
    planes = [np.asarray(Image.fromarray(np.ascontiguousarray(linear[..., c]), mode='F').resize(
        size, Image.Resampling.BOX)) for c in range(3)]
    out = np.power(np.clip(np.stack(planes, axis=-1), 0.0, 1.0), 1.0 / GAMMA)
    return np.round(out * 255.0).astype(np.uint8)


def pad_square(pixels):
    """Đệm thành ảnh vuông cạnh max(cao, rộng) bằng giá trị nhỏ nhất của từng kênh (nền của lớp lóe), ảnh ở giữa.
    Cùng phép đệm với LensFlareBank.load trong qjepa/corruptions/lens_flare.py."""
    height, width = pixels.shape[:2]
    if height == width:
        return pixels
    side = max(height, width)
    square = np.empty((side, side, 3), np.uint8)
    square[:] = pixels.min(axis=(0, 1))
    top, left = (side - height) // 2, (side - width) // 2
    square[top:top + height, left:left + width] = pixels
    return square


def fetch(part, name, size, md5):
    """Tải một ảnh, so cỡ + MD5 với bucket, thu nhỏ (hoặc giữ nguyên) rồi ghi. Trả (lỗi hoặc None, cỡ gốc)."""
    target = OUT / part / name
    error = None
    for attempt in range(RETRIES):
        try:
            with urllib.request.urlopen(f'{BUCKET}/lens-flare/{part}/{name}', timeout=120) as response:
                data = response.read()
            digest = base64.b64encode(hashlib.md5(data).digest()).decode()
            if len(data) != size or digest != md5:
                raise ValueError(f'{len(data)} byte, md5 {digest}; bucket ghi {size} byte, md5 {md5}')
            with Image.open(BytesIO(data)) as image:
                image.load()
                original = image.size
                pixels = np.asarray(image.convert('RGB'))
            temporary = target.with_suffix('.part')
            if SIDE or SQUARE:
                out = shrink(pixels, SIDE) if SIDE else pixels
                Image.fromarray(pad_square(out) if SQUARE else out).save(temporary, format='PNG')
            else:
                temporary.write_bytes(data)  # bản gốc từng byte
            os.replace(temporary, target)
            return None, original
        except Exception as exc:
            error = f'{type(exc).__name__}: {str(exc)[:300]}'
            time.sleep(5 * (attempt + 1))
    return error, None


def check_image(path):
    """(lỗi hoặc None, cỡ, chế độ màu): verify() rồi load(), vì verify() không bắt được PNG bị cắt cụt."""
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            image.load()
            return None, image.size, image.mode
    except Exception as exc:
        return f'{type(exc).__name__}: {exc}', None, None


# Chạy trong tiến trình con có ENV: liệt kê mọi file của dataset qua API, mỗi dòng một JSON.
LIST_FILES = r"""
import json, sys
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
token = None
while True:
    response = api.dataset_list_files(sys.argv[1], token, int(sys.argv[2]))
    if response is None or getattr(response, "error_message", None):
        print(json.dumps({"error": getattr(response, "error_message", None) or "no response"}))
        break
    for item in response.files or []:
        print(json.dumps({"name": item.name, "bytes": getattr(item, "total_bytes", None)}))
    token = response.next_page_token
    if not token:
        break
"""


def remote_files(dataset, env, page_size=200):
    """[{'name', 'bytes'}] của phiên bản mới nhất của dataset."""
    code, out, err = run([sys.executable, '-c', LIST_FILES, dataset, str(page_size)], env=env, timeout=1800)
    rows = [json.loads(line) for line in out.splitlines() if line.startswith('{')]
    errors = [row['error'] for row in rows if 'error' in row]
    if code != 0 or errors:
        raise RuntimeError(f'liệt kê file thất bại (mã {code}): {errors or err.strip()[-1500:]}')
    return rows


USAGE = """from pathlib import Path
from PIL import Image

# Attached datasets are mounted under /kaggle/input/<dataset> (newer notebooks may use
# /kaggle/input/datasets/<owner>/<dataset>), so look for the folder that holds captured/.
ROOT = next(path.parent for depth in range(1, 6)
            for path in sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth) + '/captured')))
captured = sorted((ROOT / 'captured').glob('*.png'))    # 2001 lab captures: ghosts, halos (frame_0000.png ...)
simulated = sorted((ROOT / 'simulated').glob('*.png'))  # 3000 wave-optics simulations: starbursts, glare
flare = Image.open(captured[0]).convert('RGB')          # flare on a dark background: add it to a clean image
                                                         # in linear light (Wu et al. use gains up to 10)
"""


def write_texts(target, counts, sizes, downloaded_at):
    """Ghi SOURCE.txt và README.md (mô tả dataset) vào ``target``; trả về README."""
    if SIDE:
        change = (f'each image was resized to a long side of {SIDE} px with a box filter in linear light (gamma 2.2), '
                  'which keeps the total flare energy')
    else:
        change = 'none in size'
    if SQUARE:
        change += ('; then padded to a square, centred, with the per-channel minimum of the image (its dark '
                   'background), so nothing is cropped or stretched')
    (target / 'SOURCE.txt').write_text(
        'Wu, He, Xue, Garg, Chen, Veeraraghavan, Barron. How to Train Neural Networks for Flare Removal. ICCV 2021.\n'
        f'gs://gresearch/lens-flare (CC BY 4.0). Changes: {change}.\n', encoding='utf-8')
    readme = f"""# {TITLE}

Flare-only images released with **Yicheng Wu, Qiurui He, Tianfan Xue, Rahul Garg, Jiawen Chen, Ashok Veeraraghavan,
Jonathan T. Barron, "How to Train Neural Networks for Flare Removal", ICCV 2021**
([paper](https://arxiv.org/abs/2011.12485), [project](https://yichengwu.github.io/flare-removal/),
[code](https://github.com/google-research/google-research/tree/master/flare_removal)).
This is an unofficial mirror; the uploader is not an author.

## License and attribution

The images are released by the authors under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) (full text in `LICENSE`). If you use them, cite the paper
above. Changes from the original (1008x752): {change}. File names are unchanged.

## Contents

| folder | images | size | what |
|---|---|---|---|
| `captured/` | {counts['captured']} | {sizes['captured']} | lab captures of a bright point source through a real lens: ghosts, halos, veiling glare (1,001 captures plus interpolated frames; consecutive frames are close) |
| `simulated/` | {counts['simulated']} | {sizes['simulated']} | wave-optics simulations of scattering flare: starbursts and glare (`apertureXXXX_blurYY_cropZZ.png`) |

All images are 8-bit RGB flare on a dark background. To make a training pair, add a flare image to a clean image in
linear light; the clean image is the target.

Downloaded from `gs://gresearch/lens-flare` on {downloaded_at}; every file was checked against the bucket's MD5.

## Usage

```python
{USAGE.rstrip()}
```
"""
    (target / 'README.md').write_text(readme, encoding='utf-8')
    return readme


def size_problems(stats):
    """Lỗi cỡ ảnh theo SIDE/SQUARE; ``stats`` = {phần: Counter('W×H mode')}."""
    problems = []
    for part, kinds in stats.items():
        for kind in kinds:
            width, height = (int(v) for v in kind.split()[0].split('×'))
            if SIDE and max(width, height) != SIDE:
                problems.append(f'{part}: có ảnh cạnh dài khác {SIDE}: {kind} ({kinds[kind]})')
            if SQUARE and width != height:
                problems.append(f'{part}: có ảnh không vuông: {kind} ({kinds[kind]})')
    return problems


print('đã định nghĩa hàm')

## Cell 3 — Môi trường và khoá Kaggle

Kiểm Internet và dung lượng trống, cài `kaggle>=2.2`, đọc secret, rồi thử đăng nhập. Khoá không được in ra.

In [ ]:
import importlib.metadata

import requests

for url in ('https://storage.googleapis.com', 'https://creativecommons.org', 'https://www.kaggle.com'):
    try:
        requests.head(url, timeout=20, allow_redirects=True)
    except requests.RequestException as error:
        raise SystemExit(f'Không vào được {url} ({type(error).__name__}). Bật Settings → Internet: On rồi chạy lại.')
print('Internet: có')
print(run(['df', '-h', '/tmp', '/kaggle/working'])[1])
need = 3e9 if SIDE is None else 1.5e9
assert shutil.disk_usage('/tmp').free > 2 * need, f'/tmp còn {gb(shutil.disk_usage("/tmp").free)}, cần ~{gb(2 * need)}'

code_, out, err = run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'kaggle>=2.2'])
assert code_ == 0, f'pip install lỗi:\n{err[-2000:]}'
print('kaggle', importlib.metadata.version('kaggle'))

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except ImportError:  # ngoài Kaggle: đọc biến môi trường
    _secrets = None
secret_errors = {}


def secret(name):
    try:
        return _secrets.get_secret(name) if _secrets else os.environ.get(name)
    except Exception as error:
        # thông báo không chứa giá trị secret; hay gặp nhất: secret chưa tạo hoặc chưa Attached cho notebook này
        secret_errors[name] = f'{type(error).__name__}: {str(error)[:300]}'
        return None


_token = secret('KAGGLE_API_TOKEN')
_key = None if _token else secret('KAGGLE_KEY')
OWNER = KAGGLE_OWNER or (None if _token else secret('KAGGLE_USERNAME'))
UPLOAD_BY_API = bool(_token or (_key and OWNER))
ENV, DATASET, EXISTS = None, None, False
if not UPLOAD_BY_API:
    for name, error in secret_errors.items():
        print(f'{name}: {error}')
    print('\nKHÔNG CÓ KHOÁ KAGGLE: notebook vẫn chạy hết. Cell 7 đóng gói dataset vào /kaggle/working; Commit xong '
          'thì tạo dataset bằng tab Output → New Dataset. Muốn tự upload: Add-ons → Secrets → KAGGLE_API_TOKEN, '
          'bật Attached cho CHÍNH notebook này, rồi chạy lại cell này.')
else:
    # Khoá chỉ nằm trong ENV, truyền cho tiến trình con kaggle. Bỏ khoá Kaggle có sẵn trong môi trường: token cũ được
    # ưu tiên hơn cặp USERNAME/KEY, sẽ đăng nhập nhầm tài khoản.
    ENV = {name: value for name, value in os.environ.items()
           if name not in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY')}
    ENV.update({'KAGGLE_API_TOKEN': _token} if _token else {'KAGGLE_USERNAME': OWNER, 'KAGGLE_KEY': _key})
del _key, _token
if UPLOAD_BY_API and not OWNER:  # tên tài khoản của token, hỏi qua API
    code_, out, err = run([sys.executable, '-c', 'from kaggle.api.kaggle_api_extended import KaggleApi; api = KaggleApi(); '
                           'api.authenticate(); print(api.get_config_value("username") or "")'], env=ENV, timeout=120)
    OWNER = (out.strip().splitlines() or [''])[-1].strip()
    if code_ != 0 or not OWNER:
        raise SystemExit(f'KAGGLE_API_TOKEN không đăng nhập được (token sai hoặc hết hạn?): {err.strip()[-500:]}')
if UPLOAD_BY_API:
    code_, text = kaggle('datasets', 'list', '--mine', '--page-size', '1', env=ENV)
    if code_ != 0 or any(word in text.lower() for word in ('401', '403', 'unauthorized', 'forbidden')):
        raise SystemExit(f'Lệnh kaggle chưa đăng nhập được (mã {code_}): {text[-500:]}')
    DATASET = f'{OWNER}/{SLUG}'
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
    EXISTS = code_ == 0 and not any(word in text.lower() for word in ('404', 'not found'))
    print(f'tài khoản Kaggle: {OWNER} | {DATASET}: '
          f'{"đã có, Cell 7 sẽ thêm version mới" if EXISTS else "chưa có, Cell 7 sẽ tạo mới"}')

## Cell 4 — Tải và kiểm MD5

Liệt kê bucket (tên, cỡ, MD5 của từng file), so số file với 2 001 / 3 000, rồi tải song song. Mỗi file được so cỡ và
MD5 với bucket trước khi giải mã; lệch hoặc lỗi mạng thì tải lại, tối đa 4 lần. File vẫn lỗi là **vấn đề** (dataset
sẽ riêng tư) và không được ghi.

In [ ]:
PROBLEMS, WARNINGS = [], []
LISTING = {part: list_part(part) for part in PARTS}
for part, rows in LISTING.items():
    print(f'bucket {part}: {len(rows)} PNG, {gb(sum(size for _, size, _ in rows))}')
    if len(rows) != EXPECTED[part]:
        PROBLEMS.append(f'bucket {part}: {len(rows)} PNG, mong đợi {EXPECTED[part]}')

if OUT.exists():
    shutil.rmtree(OUT)
for part in PARTS:
    (OUT / part).mkdir(parents=True)
started = time.perf_counter()
FAILED, ORIGINAL_SIZES = {}, Counter()
jobs = [(part, name, size, md5) for part, rows in LISTING.items() for name, size, md5 in rows]
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(fetch, *job): job for job in jobs}
    for done, future in enumerate(as_completed(futures), 1):
        part, name, _, _ = futures[future]
        error, original = future.result()
        if error:
            FAILED[f'{part}/{name}'] = error
        else:
            ORIGINAL_SIZES[part, f'{original[0]}×{original[1]}'] += 1
        if done % 500 == 0 or done == len(jobs):
            print(f'{done}/{len(jobs)} file, {len(FAILED)} lỗi, {time.perf_counter() - started:.0f} s', flush=True)
for rel, error in FAILED.items():
    print('LỖI', rel, '|', error)
if FAILED:
    PROBLEMS.append(f'{len(FAILED)} file không tải được hoặc lệch MD5 sau {RETRIES} lần: {list(FAILED)[:5]}')
leftover = list(OUT.rglob('*.part'))
for path in leftover:
    path.unlink()
DOWNLOADED_AT = time.strftime('%Y-%m-%d', time.gmtime())
print('cỡ gốc:', ', '.join(f'{part} {size} ({count})' for (part, size), count in sorted(ORIGINAL_SIZES.items())))
print(f'MD5 khớp bucket: {len(jobs) - len(FAILED)}/{len(jobs)} file | {time.perf_counter() - started:.0f} s')

## Cell 5 — Kiểm ảnh đã ghi và xem trước

Mở lại từng ảnh đã ghi (`verify()` rồi `load()`), in độ phân giải và chế độ màu của từng thư mục, so số file với số
mong đợi. Hàng giữa của hình là ảnh chụp tăng sáng ×10: bóng ma đa giác và vòng quầng của ảnh chụp tối hơn lõi
10–40 lần, nhìn ảnh thường gần như chỉ thấy lõi.

In [ ]:
import matplotlib.pyplot as plt

paths = sorted(str(path) for part in PARTS for path in (OUT / part).glob('*.png'))
with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
    CHECKED = dict(zip(paths, pool.map(check_image, paths)))
STATS, COUNTS, BYTES = {part: Counter() for part in PARTS}, Counter(), Counter()
for path, (error, size, mode) in CHECKED.items():
    part = Path(path).parent.name
    if error:
        PROBLEMS.append(f'{part}/{Path(path).name} ghi xong nhưng không đọc lại được: {error}')
        continue
    STATS[part][f'{size[0]}×{size[1]} {mode}'] += 1
    COUNTS[part] += 1
    BYTES[part] += Path(path).stat().st_size
print(f'{"thư mục":<12} {"mong đợi":>9} {"đọc được":>9} {"dung lượng":>11}  độ phân giải (số ảnh)')
for part in PARTS:
    print(f'{part:<12} {EXPECTED[part]:>9} {COUNTS[part]:>9} {gb(BYTES[part]):>11}  '
          + ', '.join(f'{kind} ({count})' for kind, count in STATS[part].most_common()))
    if COUNTS[part] != EXPECTED[part]:
        PROBLEMS.append(f'{part}: {COUNTS[part]} ảnh đọc được, mong đợi {EXPECTED[part]}')
PROBLEMS += size_problems(STATS)
for problem in PROBLEMS:
    print('VẤN ĐỀ', problem)

rng = np.random.default_rng(0)
files = {part: sorted((OUT / part).glob('*.png')) for part in PARTS}
picks = {part: [files[part][i] for i in sorted(rng.choice(len(files[part]), 4, replace=False))] for part in PARTS}
figure, axes = plt.subplots(3, 4, figsize=(16, 9))
for column in range(4):
    captured = np.asarray(Image.open(picks['captured'][column]).convert('RGB'), dtype=np.float32) / 255
    simulated = np.asarray(Image.open(picks['simulated'][column]).convert('RGB'), dtype=np.float32) / 255
    linear = np.power(captured, GAMMA)
    for row, (image, label) in enumerate(((captured, 'captured'), (np.power(np.clip(linear * 10, 0, 1), 1 / GAMMA),
                                          'captured ×10'), (simulated, 'simulated'))):
        axes[row, column].imshow(image)
        axes[row, column].set_title(f'{label}: {picks["simulated" if row == 2 else "captured"][column].name}', fontsize=8)
        axes[row, column].axis('off')
plt.tight_layout()
plt.show()

## Cell 6 — Thư mục upload

Ảnh được **chuyển** (không chép) sang `/tmp/upload/wu_flare`, kèm `LICENSE` (CC BY 4.0 nguyên văn), `SOURCE.txt`
(cùng nội dung `fetch_wu_flares` ghi), `README.md` (tiếng Anh, vì là trang dataset, cũng là phần mô tả trên Kaggle)
và `dataset-metadata.json` (CLI chỉ đọc, không upload thành file).

In [ ]:
if UPLOAD.exists():
    shutil.rmtree(UPLOAD)
UPLOAD.mkdir(parents=True)
for part in PARTS:
    shutil.move(str(OUT / part), UPLOAD / part)  # cùng ổ /tmp: chỉ đổi tên
FINAL = {part: sorted(path.name for path in (UPLOAD / part).iterdir()) for part in PARTS}
assert {part: len(names) for part, names in FINAL.items()} == {part: COUNTS[part] for part in PARTS}, \
    'số file sau khi chuyển khác lúc kiểm'

response = requests.get(LICENSE_URL, timeout=60)
response.raise_for_status()
assert response.content.startswith(b'Attribution 4.0 International'), f'{LICENSE_URL} không phải CC BY 4.0'
(UPLOAD / 'LICENSE').write_bytes(response.content)
upstream = requests.get(README_URL, timeout=60).text
if 'CC BY 4.0' not in upstream:
    WARNINGS.append(f'README của flare_removal không còn ghi "CC BY 4.0" ({README_URL}): kiểm lại giấy phép')
sizes = {part: ', '.join(f'{kind} ({count})' for kind, count in STATS[part].most_common()) for part in PARTS}
README = write_texts(UPLOAD, COUNTS, sizes, DOWNLOADED_AT)
assert 6 <= len(TITLE) <= 50 and 20 <= len(SUBTITLE) <= 80, 'Kaggle: title 6–50 ký tự, subtitle 20–80 ký tự'
VISIBLE_PUBLIC = PUBLIC and not PROBLEMS
if UPLOAD_BY_API:
    (UPLOAD / 'dataset-metadata.json').write_text(json.dumps(
        {'title': TITLE, 'id': DATASET, 'licenses': [{'name': 'CC-BY-4.0'}], 'subtitle': SUBTITLE,
         'description': README}, indent=2, ensure_ascii=False))
total = sum(path.stat().st_size for path in UPLOAD.rglob('*') if path.is_file())
print(f'{UPLOAD}: {sum(map(len, FINAL.values()))} ảnh + LICENSE, SOURCE.txt, README.md | {gb(total)}')
print('dataset sẽ', 'PUBLIC' if VISIBLE_PUBLIC else 'RIÊNG TƯ',
      f'(còn {len(PROBLEMS)} vấn đề: {PROBLEMS})' if PROBLEMS and PUBLIC else '')
for warning in WARNINGS:
    print('CẢNH BÁO', warning)

## Cell 7 — Upload

**Có khoá:** `kaggle datasets create -p /tmp/upload/wu_flare --dir-mode zip`; dataset đã có thì thêm version mới.
Kaggle CLI có lúc in lỗi mà vẫn trả mã 0, và file upload lỗi chỉ in `Upload unsuccessful` rồi dataset vẫn được tạo
thiếu file (đo khi dựng HALO, Flare7K++), nên cell đọc cả chữ trong output.

**Không có khoá:** mỗi thư mục thành một tar không nén trong `/kaggle/working` (*New Dataset* từ Output chỉ nhận 500
file đầu), kèm `LICENSE`, `SOURCE.txt`, `README.md`. Kaggle giải nén `captured.tar` thành `captured/`.

In [ ]:
import tarfile
import tempfile

PACKED = {}
if UPLOAD_BY_API:
    largest = max(sum(path.stat().st_size for path in (UPLOAD / part).iterdir()) for part in PARTS)
    free = shutil.disk_usage(tempfile.gettempdir()).free
    assert free > largest * 1.05 + 1e9, f'{tempfile.gettempdir()} còn {gb(free)}, zip tạm lớn nhất cần {gb(largest)}'
    if EXISTS:
        args = ['datasets', 'version', '-p', str(UPLOAD), '--dir-mode', 'zip', '-m', f'gs://gresearch/lens-flare {DOWNLOADED_AT}']
        print('Dataset đã có: thêm version mới (chế độ hiển thị giữ như cũ)')
    else:
        args = ['datasets', 'create', '-p', str(UPLOAD), '--dir-mode', 'zip'] + (['--public'] if VISIBLE_PUBLIC else [])
    started = time.perf_counter()
    code_, text = kaggle(*args, env=ENV, timeout=4 * 3600)
    print(text[-4000:])
    print(f'{time.perf_counter() - started:.0f} s')
    failed = [word for word in ('error', 'unsuccessful', 'skipping') if word in text.lower()]
    assert code_ == 0 and not failed, f'Upload thất bại (mã {code_}, thấy {failed}); xem output ở trên'
    UPLOADED = True  # Cell 9 bỏ qua
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    total = sum(path.stat().st_size for path in UPLOAD.rglob('*') if path.is_file())
    free = shutil.disk_usage(OUTPUT_DIR).free
    assert free > total + 1e9 and total < 19e9, f'{OUTPUT_DIR} còn {gb(free)}, cần {gb(total)} (Kaggle lưu tối đa 20 GB)'
    for part in PARTS:
        target, partial = OUTPUT_DIR / f'{part}.tar', OUTPUT_DIR / f'{part}.tar.part'
        written = {}
        with tarfile.open(partial, 'w', format=tarfile.PAX_FORMAT) as tar:
            for path in sorted((UPLOAD / part).iterdir()):
                tar.add(path, arcname=path.name, recursive=False)
                written[path.name] = path.stat().st_size
        with tarfile.open(partial) as tar:
            read_back = {member.name: member.size for member in tar if member.isfile()}
        assert read_back == written, f'{partial}: tar đọc lại khác lúc ghi'
        os.replace(partial, target)
        PACKED[part] = len(written)
        print(f'{target.name}: {len(written)} file, {gb(target.stat().st_size)}, đọc lại khớp')
    for name in ('LICENSE', 'SOURCE.txt', 'README.md'):
        shutil.copy2(UPLOAD / name, OUTPUT_DIR / name)
    print(f"""
Tạo dataset không cần tải lại: gắn secret KAGGLE_API_TOKEN, chạy Cell 1, 2, 3 rồi Cell 9.
Hoặc tạo tay, Commit xong:
  1. Mở trang phiên bản vừa Commit → tab Output → New Dataset.
  2. Tên: {SLUG}. Chọn đủ captured.tar, simulated.tar, LICENSE, SOURCE.txt, README.md.
  3. Đặt {"Public" if VISIBLE_PUBLIC else "Private"}; License: CC BY 4.0; dán README.md làm mô tả.
  4. Chờ Kaggle giải nén: trang dataset phải có captured/ ({PACKED.get('captured')} file) và simulated/ ({PACKED.get('simulated')} file).""")

## Cell 8 — Xác minh trên Kaggle, dọn dẹp, báo cáo

Chờ trạng thái `ready`, liệt kê mọi file của dataset qua API, so số file từng thư mục với lúc upload, rồi xoá dữ liệu
trong `/tmp` và biến chứa khoá. Báo cáo ghi vào `/kaggle/working/wu_flare_report.md` (chỉ khi upload bằng khoá; tạo
tay từ Output thì `/kaggle/working` chính là dataset nên không ghi thêm gì vào đó).

In [ ]:
status, started = ('', time.perf_counter()) if UPLOAD_BY_API else ('chưa tạo: tạo tay từ tab Output', 0)
for _ in range(120 if UPLOAD_BY_API else 0):  # Kaggle giải nén các zip sau khi nhận xong; chờ tối đa 60 phút
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
    status = (text.splitlines() or [''])[-1].strip().lower()
    if status in ('ready', 'error'):
        break
    time.sleep(30)
REMOTE = remote_files(DATASET, ENV) if status == 'ready' else []
remote_counts = Counter(posixpath.dirname(row['name']) or '.' for row in REMOTE)
local_counts = Counter({part: len(names) for part, names in FINAL.items()})
local_counts['.'] = 3  # LICENSE, SOURCE.txt, README.md
VERIFY = [(folder, local_counts[folder], remote_counts[folder] if REMOTE else '-')
          for folder in sorted(set(local_counts) | set(remote_counts))]
MISMATCHED = [row for row in VERIFY if REMOTE and row[1] != row[2]]
if REMOTE:
    print(f'{"thư mục":<12} {"local":>7} {"Kaggle":>7}')
    for folder, here, there in VERIFY:
        print(f'{folder:<12} {here:>7} {there:>7}{"" if here == there else "   <- LỆCH"}')

shutil.rmtree(OUT, ignore_errors=True)
shutil.rmtree(UPLOAD.parent, ignore_errors=True)
del ENV  # khoá chỉ từng nằm trong biến này

if UPLOAD_BY_API:
    link = f'https://www.kaggle.com/datasets/{DATASET}'
    visibility = 'như version trước (dataset đã có)' if EXISTS else 'public' if VISIBLE_PUBLIC else 'riêng tư'
else:
    link = f'chưa tạo: tab Output → New Dataset, tên {SLUG}'
    visibility = 'nên đặt public' if VISIBLE_PUBLIC else 'nên giữ riêng tư'
lines = [
    '# Ảnh lóe Wu et al. → Kaggle: báo cáo', '',
    f'- Dataset: {link} ({visibility}); trạng thái Kaggle: {status}',
    f'- Nguồn: gs://gresearch/lens-flare, tải {DOWNLOADED_AT}, MD5 từng file khớp bucket',
    f'- Thu nhỏ: {"cạnh dài " + str(SIDE) + " px, BOX trên ánh sáng tuyến tính" if SIDE else "không"}'
    f'{"; đệm vuông bằng nền của ảnh" if SQUARE else ""}', '',
    '| thư mục | mong đợi | local | Kaggle | độ phân giải |', '|---|---|---|---|---|',
]
remote_of = {folder: there for folder, _, there in VERIFY}
lines += [f'| `{part}` | {EXPECTED[part]} | {COUNTS[part]} | {remote_of.get(part, "-")} | {sizes[part]} |' for part in PARTS]
lines += ['', '## Vấn đề', ''] + ([f'- {problem}' for problem in PROBLEMS] or ['- không có'])
lines += ['', '## Cảnh báo', ''] + ([f'- {warning}' for warning in WARNINGS] or ['- không có'])
lines += ['', '## Đối chiếu sau upload', '']
lines += [f'- `{folder}`: local {here}, Kaggle {there}' for folder, here, there in MISMATCHED] or (
    ['- khớp từng thư mục'] if REMOTE else ['- chưa liệt kê được file trên Kaggle' if UPLOAD_BY_API else
                                         '- bỏ qua: dataset tạo tay từ Output, kiểm trên trang dataset'])
lines += ['', '## Đọc dữ liệu trong notebook train', '', '```python', USAGE.rstrip(), '```', '']
REPORT_TEXT = '\n'.join(lines)
if UPLOAD_BY_API:
    REPORT.parent.mkdir(parents=True, exist_ok=True)
    REPORT.write_text(REPORT_TEXT, encoding='utf-8')
print(REPORT_TEXT)
assert not MISMATCHED, f'{len(MISMATCHED)} thư mục lệch số file giữa local và Kaggle'

## Cell 9 — Tạo dataset từ gói đã đóng (khi lần chạy trước không có khoá)

Lần chạy không có khoá để lại gói trong `/kaggle/working`: `captured.tar`, `simulated.tar`, `LICENSE`, `SOURCE.txt`,
`README.md`. Cell này tạo dataset từ gói đó, **không tải lại ảnh**:

1. *Add-ons → Secrets*: **`KAGGLE_API_TOKEN`**, bật *Attached* cho notebook này.
2. Chạy **Cell 1, Cell 2, Cell 3** (chỉ định nghĩa và đọc khoá, không tải gì), rồi **cell này**.

Cell tìm gói trong `/kaggle/working` (cùng phiên vừa chạy), rồi trong `/kaggle/input` (Output của phiên bản đã Commit,
gắn làm Input: Kaggle có thể đã giải nén tar thành thư mục `captured/`, `simulated/`). Cell giải nén vào
`/tmp/upload/wu_flare`, đếm lại 2 001 / 3 000 PNG, **đệm ảnh chưa vuông thành 640×640** (gói của lần chạy cũ là
640×477; cùng phép đệm như Cell 4, không tải lại), ghi lại `README.md`/`SOURCE.txt` cho đúng cỡ mới, tạo dataset (riêng tư nếu `PUBLIC = False`; đã có thì thêm version
mới), chờ `ready` rồi so số file trên Kaggle. Khi Run All đã upload bằng khoá ở Cell 7 thì cell này bỏ qua.

In [ ]:
import re
import tarfile


def find_package(depth=5):
    """Thư mục có README.md và, với mỗi phần, <phần>.tar hoặc thư mục <phần>/: /kaggle/working trước, rồi Input."""
    def complete(base):
        return (base / 'README.md').is_file() and all(
            (base / f'{part}.tar').is_file() or (base / part).is_dir() for part in PARTS)
    if complete(OUTPUT_DIR):
        return OUTPUT_DIR
    for level in range(depth):
        for readme in sorted(Path('/kaggle/input').glob('/'.join(['*'] * level + ['SOURCE.txt']))):
            if complete(readme.parent):
                return readme.parent
    return None


if globals().get('UPLOADED'):
    print(f'Phiên này đã upload {DATASET} bằng khoá (Cell 7 hoặc cell này): bỏ qua.')
elif not UPLOAD_BY_API:
    print('Chưa có khoá: Add-ons → Secrets → KAGGLE_API_TOKEN, bật Attached cho notebook này, chạy Cell 1, 2, 3 rồi '
          'chạy lại cell này. (Hoặc tạo tay: tab Output → New Dataset, như Cell 7 hướng dẫn.)')
else:
    PACKAGE = find_package()
    assert PACKAGE, ('Không thấy gói (README.md + captured.tar/simulated.tar hoặc captured/ simulated/) trong '
                     '/kaggle/working hay /kaggle/input. Gắn Output của phiên bản đã Commit làm Input, hoặc chạy cả notebook.')
    print(f'gói: {PACKAGE}')
    if UPLOAD.exists():
        shutil.rmtree(UPLOAD)
    PACKAGE_COUNTS, PACKAGE_PROBLEMS = {}, []
    for part in PARTS:
        (UPLOAD / part).mkdir(parents=True)
        archive = PACKAGE / f'{part}.tar'
        if archive.is_file():
            with tarfile.open(archive) as tar:
                members = [member for member in tar if member.isfile()]
                unsafe = [member.name for member in members if '/' in member.name or member.name.startswith('.')]
                assert not unsafe, f'{archive}: tên file lạ trong tar {unsafe[:5]}'
                for member in members:
                    with tar.extractfile(member) as source, open(UPLOAD / part / member.name, 'wb') as target:
                        shutil.copyfileobj(source, target)
        else:
            for path in sorted((PACKAGE / part).glob('*.png')):
                shutil.copy2(path, UPLOAD / part / path.name)
        PACKAGE_COUNTS[part] = sum(1 for path in (UPLOAD / part).iterdir() if path.suffix == '.png')
        if PACKAGE_COUNTS[part] != EXPECTED[part]:
            PACKAGE_PROBLEMS.append(f'{part}: {PACKAGE_COUNTS[part]} PNG, mong đợi {EXPECTED[part]}')

    def square_one(path):
        """Đệm vuông một PNG tại chỗ nếu cần; trả (cỡ sau, chế độ màu, đã đệm?) hoặc lỗi."""
        try:
            with Image.open(path) as image:
                image.load()
                size, mode = image.size, image.mode
                pixels = np.asarray(image.convert('RGB')) if SQUARE and size[0] != size[1] else None
            if pixels is None:
                return size, mode, False
            out = pad_square(pixels)
            temporary = path.with_suffix('.part')
            Image.fromarray(out).save(temporary, format='PNG')
            os.replace(temporary, path)
            return (out.shape[1], out.shape[0]), 'RGB', True
        except Exception as exc:
            return f'{type(exc).__name__}: {exc}'

    PACKAGE_STATS, padded = {part: Counter() for part in PARTS}, 0
    for part in PARTS:
        pngs = sorted((UPLOAD / part).glob('*.png'))
        with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
            for path, result in zip(pngs, pool.map(square_one, pngs)):
                if isinstance(result, str):
                    PACKAGE_PROBLEMS.append(f'{part}/{path.name} không đọc được: {result}')
                    continue
                (width, height), mode, changed = result
                PACKAGE_STATS[part][f'{width}×{height} {mode}'] += 1
                padded += changed
    PACKAGE_PROBLEMS += size_problems(PACKAGE_STATS)
    print(f'đệm vuông {padded} ảnh |', ', '.join(f'{part}: ' + ', '.join(f'{kind} ({count})' for kind, count in
                                                      PACKAGE_STATS[part].most_common()) for part in PARTS))

    old_readme = (PACKAGE / 'README.md').read_text(encoding='utf-8', errors='replace')
    found = re.search(r'on (\d{4}-\d{2}-\d{2}); every file was checked', old_readme)
    package_downloaded_at = found.group(1) if found else 'unknown date'
    if (PACKAGE / 'LICENSE').is_file():
        shutil.copy2(PACKAGE / 'LICENSE', UPLOAD / 'LICENSE')
    else:
        PACKAGE_PROBLEMS.append('thiếu LICENSE')
    package_sizes = {part: ', '.join(f'{kind} ({count})' for kind, count in PACKAGE_STATS[part].most_common())
                     for part in PARTS}
    write_texts(UPLOAD, PACKAGE_COUNTS, package_sizes, package_downloaded_at)
    print('PNG:', PACKAGE_COUNTS, '|', gb(sum(path.stat().st_size for path in UPLOAD.rglob('*') if path.is_file())))
    for problem in PACKAGE_PROBLEMS:
        print('VẤN ĐỀ', problem)
    assert not any('không vuông' in problem or 'cạnh dài' in problem for problem in PACKAGE_PROBLEMS), \
        'ảnh chưa đúng cỡ: không upload'

    PACKAGE_PUBLIC = PUBLIC and not PACKAGE_PROBLEMS
    (UPLOAD / 'dataset-metadata.json').write_text(json.dumps(
        {'title': TITLE, 'id': DATASET, 'licenses': [{'name': 'CC-BY-4.0'}], 'subtitle': SUBTITLE,
         'description': (UPLOAD / 'README.md').read_text(encoding='utf-8')}, indent=2, ensure_ascii=False))
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)  # hỏi lại: có thể đã tạo sau khi chạy Cell 3
    package_exists = code_ == 0 and not any(word in text.lower() for word in ('404', 'not found'))
    if package_exists:
        args = ['datasets', 'version', '-p', str(UPLOAD), '--dir-mode', 'zip', '-m', 'from the packaged builder output']
        print(f'{DATASET} đã có: thêm version mới (chế độ hiển thị giữ như cũ)')
    else:
        args = ['datasets', 'create', '-p', str(UPLOAD), '--dir-mode', 'zip'] + (['--public'] if PACKAGE_PUBLIC else [])
        print(f'tạo {DATASET} ({"public" if PACKAGE_PUBLIC else "riêng tư"})')
    started = time.perf_counter()
    code_, text = kaggle(*args, env=ENV, timeout=4 * 3600)
    print(text[-4000:])
    print(f'{time.perf_counter() - started:.0f} s')
    # CLI có lúc in lỗi mà vẫn trả mã 0; file upload lỗi chỉ in "Upload unsuccessful" rồi tạo dataset thiếu file đó
    failed = [word for word in ('error', 'unsuccessful', 'skipping') if word in text.lower()]
    assert code_ == 0 and not failed, f'Upload thất bại (mã {code_}, thấy {failed}); xem output ở trên'
    UPLOADED = True

    status, started = '', time.perf_counter()
    for _ in range(120):  # Kaggle giải nén các zip sau khi nhận xong; chờ tối đa 60 phút
        code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
        status = (text.splitlines() or [''])[-1].strip().lower()
        if status in ('ready', 'error'):
            break
        time.sleep(30)
    print(f'trạng thái: {status} sau {(time.perf_counter() - started) / 60:.0f} phút')
    REMOTE = remote_files(DATASET, ENV) if status == 'ready' else []
    remote_counts = Counter(posixpath.dirname(row['name']) or '.' for row in REMOTE)
    local_counts = Counter(PACKAGE_COUNTS)
    local_counts['.'] = sum((UPLOAD / name).is_file() for name in ('LICENSE', 'SOURCE.txt', 'README.md'))
    MISMATCHED = [(folder, local_counts[folder], remote_counts[folder])
                  for folder in sorted(set(local_counts) | set(remote_counts)) if local_counts[folder] != remote_counts[folder]]
    shutil.rmtree(UPLOAD.parent, ignore_errors=True)
    print(f'https://www.kaggle.com/datasets/{DATASET} | Kaggle: {len(REMOTE)} file | '
          + ('khớp từng thư mục' if REMOTE and not MISMATCHED else 'chưa liệt kê được file' if not REMOTE else
             'LỆCH: ' + ', '.join(f'{folder} local {here} Kaggle {there}' for folder, here, there in MISMATCHED)))
    assert REMOTE and not MISMATCHED, 'dataset trên Kaggle chưa khớp gói; xem dòng trên'

## Dùng trong notebook train

Gắn dataset vào `qwt-jaco-jepa-ijepa.ipynb` (*Add Input*). Cell 4 gọi `find_wu_flares(INPUT)`: thư mục nào có
`captured/` (≥ 2 001 PNG) và `simulated/` (≥ 3 000 PNG) thì dùng luôn, không tải lại. Với `SIDE = 640` ảnh là bản
`fetch_wu_flares(side=640)` tự tải, đệm vuông đúng như `LensFlareBank.load` đệm khi nạp, nên lớp lóe trùng từng pixel và
kết quả train không đổi. `data.wu_flare_root` nằm ngoài mọi
hash, nên đổi từ bản tự tải sang dataset không làm mất hiệu lực checkpoint nào.